# Занятие 34. Практика: bagging и случайный лес (~90 мин)

Вы **пишете весь код сами**. Ячейку **«Дано»** не меняйте.

Главная модель — **RandomForestClassifier** (теория — занятие 33).

Сравним одно дерево, bagging и random forest; настроим `n_estimators`, посмотрим OOB-оценку (out-of-bag).

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|---|---:|
| 0 | Split | 3 |
| 1 | Три модели | 4 |
| 2 | n_estimators | 4 |
| 3 | OOB-оценка | 3 |
| 4 | max_depth | 4 |
| 5 | Bootstrap вручную | 3 |
| 6 | Permutation importance | 5 |
| 7 | Разрыв между train и validation | 2 |
| 8 | Итог | 2 |

---
## Дано: make_classification

20 признаков — как в теории.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=7, flip_y=0.08, random_state=42
)
print('Объектов:', len(X))

---
## Задание 0. Split (~8 мин) — **3 балла**

Подготовьте инструменты для сравнения ансамблей. Импортируйте `train_test_split`, `DecisionTreeClassifier`, `BaggingClassifier`, `RandomForestClassifier`, `accuracy_score` и `permutation_importance`.

Разделите данные на train и validation в пропорции 70/30, используйте `stratify=y` и `RANDOM_STATE=42`. Стратификация нужна, чтобы доли классов в обеих частях были похожи, а сравнение моделей было честнее.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — импортированы нужные модели и метрики.
- **1.0 балл** — выполнен split 70/30 с stratify=y.
- **1.0 балл** — задан и используется RANDOM_STATE=42.

### Снижение баллов
- Нет stratify → минус **0.5**.
- Нет отдельной validation-части → минус **1.0**.

In [ ]:
RANDOM_STATE = 42
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE,
)
print(len(X_train), len(X_val))

---
## Задание 1. Три модели (~15 мин) — **4 балла**

Обучите три модели:

1. одно решающее дерево;
2. `BaggingClassifier` с 150 базовыми моделями;
3. `RandomForestClassifier` с 150 деревьями.

Для каждой модели посчитайте train и validation accuracy.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучено одно дерево.
- **1.0 балл** — обучен BaggingClassifier.
- **1.0 балл** — обучен RandomForestClassifier.
- **1.0 балл** — есть таблица train/validation accuracy для всех трёх моделей.

### Снижение баллов
- Нет train или validation accuracy → минус **0.5**.
- Модели сравниваются на разных выборках → минус **1.0**.

In [ ]:
models = {
    'tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'bagging': BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'rf': RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, accuracy_score(y_train, m.predict(X_train)),
          accuracy_score(y_val, m.predict(X_val)))

---
## Задание 2. n_estimators (~15 мин) — **4 балла**

Проверьте, как количество деревьев влияет на качество случайного леса. Обучите `RandomForestClassifier` для значений `n_estimators` из списка `[1, 5, 20, 60, 150, 300]` и каждый раз измерьте validation accuracy.

Постройте график «число деревьев → validation accuracy». По графику найдите область, где качество перестаёт заметно расти: это поможет увидеть, когда добавление новых деревьев уже почти не окупается.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — проверены все значения n_estimators.
- **1.0 балл** — для каждого значения посчитана validation accuracy.
- **1.0 балл** — построен график n_estimators → validation accuracy.
- **1.0 балл** — есть вывод о плато качества.

### Снижение баллов
- График отсутствует → минус **0.5**.
- Вывод о плато не сделан → минус **0.5**.

In [ ]:
ns = [1, 5, 20, 60, 150, 300]
val_scores = []
for n in ns:
    m = RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train, y_train)
    val_scores.append(accuracy_score(y_val, m.predict(X_val)))
plt.plot(ns, val_scores, 'o-')
plt.xlabel('n_estimators')
plt.ylabel('validation accuracy')
plt.title('Плато качества леса')
plt.show()

---
## Задание 3. OOB-оценка (~10 мин) — **3 балла**

Обучите `RandomForestClassifier(oob_score=True, n_estimators=300)`. OOB (out-of-bag) — это оценка качества на тех train-объектах, которые не попали в bootstrap-выборку конкретных деревьев.

Выведите OOB-оценку и accuracy на validation, затем сравните их. Если значения близки, OOB можно воспринимать как дополнительную внутреннюю проверку качества леса; если сильно различаются, стоит подумать о размере выборки, шуме и настройках модели.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучен лес с oob_score=True.
- **1.0 балл** — выведена OOB-оценка.
- **1.0 балл** — OOB-оценка сравнена с validation accuracy.

### Снижение баллов
- Не включён oob_score=True → минус **1.0**.
- Нет сравнения с validation → минус **0.5**.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)
print('OOB-оценка:', rf.oob_score_)
print('validation accuracy:', accuracy_score(y_val, rf.predict(X_val)))

**Вывод:**

OOB-оценка должна быть близка к accuracy на validation, потому что она тоже оценивает качество на объектах, не использованных конкретными деревьями при обучении. Если OOB и validation сильно расходятся, нужно проверить размер данных, случайность split и устойчивость модели.


---
## Задание 4. max_depth (~12 мин) — **4 балла**

Проверьте значения `max_depth` из списка `[None, 5, 10, 20]` при `n_estimators=150`. Выберите лучший вариант по validation accuracy.

Если несколько вариантов дают одинаковое качество, выберите более простую модель: меньшую глубину.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — проверены все значения max_depth.
- **1.0 балл** — при сравнении зафиксирован n_estimators=150.
- **1.0 балл** — выбран best_depth по validation accuracy.
- **1.0 балл** — при равенстве качества выбран более простой вариант.

### Снижение баллов
- best_depth выбран по train accuracy → минус **1.0**.
- Нет таблицы сравнения → минус **0.5**.

In [ ]:
best_depth, best_acc = None, -1
for d in [None, 5, 10, 20]:
    m = RandomForestClassifier(
        n_estimators=150, max_depth=d, random_state=RANDOM_STATE, n_jobs=-1,
    )
    m.fit(X_train, y_train)
    acc = accuracy_score(y_val, m.predict(X_val))
    print(d, acc)
    if acc > best_acc:
        best_acc, best_depth = acc, d
print('best_depth:', best_depth)

---
## Задание 5. Bootstrap вручную (~12 мин) — **3 балла**

Сделайте 100 повторов bootstrap длины `len(X_train)`.

Bootstrap означает: случайно выбираем индексы объектов **с возвращением**, поэтому один объект может попасть несколько раз, а другой — ни разу.

В каждом повторе посчитайте долю уникальных индексов, затем найдите среднюю долю уникальных.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — выполнено 100 bootstrap-повторов.
- **1.0 балл** — выборка строится с возвращением.
- **1.0 балл** — посчитана средняя доля уникальных объектов.

### Снижение баллов
- Индексы выбираются без возвращения → минус **1.0**.
- Нет среднего значения доли уникальных объектов → минус **0.5**.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = len(X_train)
fracs = [len(np.unique(rng.choice(n, n, replace=True))) / n for _ in range(100)]
print('средняя доля уникальных:', round(np.mean(fracs), 3))

---
## Задание 6. Permutation importance — важность через перемешивание (~15 мин)

Посчитайте permutation importance на validation для лучшего `RandomForestClassifier` из предыдущих заданий. Идея такая: по очереди перемешиваем один признак и смотрим, насколько падает качество модели.

Выведите top-5 признаков: таблицей или горизонтальным столбчатым графиком (`barh`).

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучен выбранный RandomForestClassifier.
- **1.0 балл** — permutation_importance посчитан на validation.
- **1.0 балл** — показаны top-5 признаков.
- **1.0 балл** — важности представлены таблицей или графиком.
- **1.0 балл** — понятно, для какой модели и метрики считалась важность.

### Снижение баллов
- Importance посчитана на train без объяснения → минус **0.5**.
- Нет top-5 признаков → минус **0.5**.

In [ ]:
rf = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
r = permutation_importance(rf, X_val, y_val, n_repeats=10, random_state=RANDOM_STATE)
imp = r.importances_mean
top = np.argsort(imp)[-5:]
for i in top[::-1]:
    print(f'f{i}', round(imp[i], 4))

---
## Задание 7. Разрыв между train и validation (~8 мин) — **2 балла**

В markdown объясните: почему высокий train accuracy у случайного леса сам по себе ещё не доказывает переобучение?

Подсказка: смотреть нужно не только на train, но и на validation/OOB-оценку. Если validation accuracy или OOB-оценка тоже хорошие, высокий train accuracy менее тревожен.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — объяснение сравнивает train accuracy с validation или OOB.
- **1.0 балл** — вывод написан в 2–3 предложениях и связан с результатами практики.

### Снижение баллов
- Высокий train accuracy автоматически назван переобучением без проверки validation/OOB → минус **0.5**.

**Ответ:**

Высокий train accuracy у случайного леса сам по себе не доказывает переобучение: ансамбль действительно может почти идеально описывать train. Диагноз появляется по разрыву с validation или OOB-оценкой. Если validation accuracy и OOB-оценка тоже высокие и близки к train, модель, скорее всего, обобщает нормально.


---
## Задание 8. Итог (~5 мин) — **2 балла**

Напишите два коротких вывода: один про bagging, второй про random forest. В выводе про bagging объясните, зачем обучать много деревьев на bootstrap-выборках. В выводе про random forest добавьте, чем он отличается от обычного bagging над деревьями.

Постарайтесь связать формулировки с результатами практики: качеством одного дерева, bagging и случайного леса на validation.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — корректно описан bagging.
- **1.0 балл** — корректно описано отличие random forest от bagging.

### Снижение баллов
- Нет различия между bagging и random forest → минус **0.5**.

**Ответ:**

Bagging обучает много деревьев на разных bootstrap-выборках и усредняет их ответы, снижая разброс относительно одного дерева.

Random Forest делает то же самое, но дополнительно случайно выбирает подмножество признаков в узлах, поэтому деревья становятся разнообразнее, а ансамбль часто работает устойчивее.
